In [1]:
import pandas as pd
import numpy as np
import re, os

from tqdm.notebook import tqdm
from collections import defaultdict

import importlib
import ask_mode as am, mode_helper_V4
importlib.reload(am)
from ask_mode import *
importlib.reload(mode_helper_V4)
from mode_helper_V4 import *
from xml.etree import ElementTree as ET

In [2]:
def get_label_records_setid(set_id, spl_cache=False):
    df_result = pd.DataFrame(columns= ['SETID', 'SPL_ID', 'Product Name', 'Company', 
                'SPL Effective Time', 'Revised Date','Initial Year', 'XML', 'Document_Type'])
    
    if spl_cache:
        xml_file = 'References_NEW/'+set_id+'.xml'
        xml_data = ET.tostring(ET.parse(xml_file).getroot(), method='xml').decode('ISO8859-1')
        results = [[set_id, '', '', '', '', '', '', xml_data, '', set_id]]
        df_result = pd.DataFrame(results, columns= ['SETID', 'SPL_ID', 'Product Name', 'Company', 
                'SPL Effective Time', 'Revised Date','Initial Year', 'XML', 'Document_Type', 'Used_Name'])
        return df_result
    
    # first use exact match if the drugname is specified
    if set_id:
        # the labeling used has to include the Adverse Reaction section; this is a simple checker to ensure the labeling document is valid to use .
        query_exact = f"""
            select l.set_id, l.spl_id, l.product_names, l.AUTHOR_ORG_NORMD_NAME, 
                l.eff_time, l.revised_date, 
                l.initial_approval_year, s.spl_xml, l.document_type_loinc_code
                from druglabel.sum_spl l
                join druglabel.spl s on s.set_id = l.set_id
                where l.set_id = '{set_id}'
                order by l.eff_time desc
            """
        
        results = cursor.execute(query_exact).fetchmany(top) # Only consider the top 1 labeling
        df_result = pd.DataFrame(results, columns= ['SETID', 'SPL_ID', 'Product Name', 'Company', 
                'SPL Effective Time', 'Revised Date','Initial Year', 'XML', 'Document_Type'])
        if df_result.shape[0]:
            df_result['Used_Name'] = set_id
            return df_result
    else:
        return df_result
        # print(query_exact, df_result.shape)

In [3]:
def main(df_run, drug_col, prompt, mode='M2', setid_use=False, spl_cache=False, top=3):
    for ind, d in tqdm(df_run.iterrows()):
        if mode == 'M2':
            tox_class = 'DILI'
        elif mode == 'M4':
            tox_class = 'DICT'
        elif mode == 'M5':
            tox_class = 'DIRI'
        else:
            print(mode, 'does not support!')
            break

        # visited items
        if d['Toxicity Class']:
            continue
            
        drugname = d[drug_col].strip()
        message = f'What is the {tox_class} Class of {drugname}?'
        if mode in ('M2', 'M5'):
            section_used = ['34066-1','43685-7','34084-4', '34071-1', '42232-9']
        elif mode == 'M4':
            section_used = ['34066-1','43685-7','34084-4', '34071-1', '42232-9', '34088-5']
        
        if setid_use is True:
            setid = d['SETID'].strip()
            if setid:
                df_labeling = get_label_records_setid(set_id=setid, spl_cache=spl_cache)
            else:
                df_labeling = get_label_records(message, mode=mode, drugname=drugname, top=top)
            # print(df_labeling.shape)
        else:
            df_labeling = get_label_records(message, mode=mode, drugname=drugname, top=top)
        
        meta_ask = {'name_list':drugname, 'mode':mode, 'section_used':section_used, 'max_token':256, 'df_labeling':df_labeling}    
        df_answer = ask_mode(message, prompt, meta_ask)

        if not df_answer.shape[0]: # no labeling section was found and analyzed. skip this item.
            df_run.loc[ind, 'Toxicity Class'] = 'Not Available'
            df_run.loc[ind, 'Evidence'] = 'No Labeling was found to support the query.'
            df_run.loc[ind, 'Supported Section'] = 'N/A'
            continue
            
        # df_run.loc[ind, 'raw_output'] = df_answer.to_json()

        ## check the boxed warnings section first.
        BW_responses = df_answer[df_answer['curr_section']=='34066-1']
        for _, d_tmp in BW_responses.iterrows():
            if '(Found' in d_tmp['section_answer'] and not (re.search('None', d_tmp['section_answer'], re.I)):
                df_run.loc[ind, 'Toxicity Class'] = 'Most'
                df_run.loc[ind, 'Evidence'] = d_tmp['section_answer']
                df_run.loc[ind, 'Supported Section'] = d_tmp['curr_section']
                df_run.loc[ind, 'SETID'] = d_tmp['reference']['SETID']
                break
    
        ## if no result, check the warnings and precautions section next.
        if df_run.loc[ind, 'Toxicity Class'] in ('Most', 'Less'):
            continue
        WP_responses = df_answer[df_answer['curr_section'].isin(['43685-7', '34071-1', '42232-9'])]
        for _, d_tmp in WP_responses.iterrows():
            if '(Found' in d_tmp['section_answer']:
                if (mode == 'M2'):
                    Score = re.findall(r'(?<=Score: )(\d+)', d_tmp['section_answer'])
                    if not Score: # no Severity score mentioned, skip this disqualified item
                        continue
                    else:
                        Score = max([int(x) for x in Score])
                        if Score > 3:
                            df_run.loc[ind, 'Toxicity Class'] = 'Most'
                        elif Score == 0:
                            df_run.loc[ind, 'Toxicity Class'] = 'Precaution'
                        else:
                            df_run.loc[ind, 'Toxicity Class'] = 'Less'        
                elif mode in ('M4', ):
                    if 'Severe' in d_tmp['section_answer'] or 'Moderate' in d_tmp['section_answer']:
                        df_run.loc[ind, 'Toxicity Class'] = 'Most'
                    elif 'Mild' in d_tmp['section_answer']:
                        df_run.loc[ind, 'Toxicity Class'] = 'Less'
                    elif re.search('Pre-existed', d_tmp['section_answer'], re.I):
                        df_run.loc[ind, 'Toxicity Class'] = 'Precaution'
                    else:
                        continue
                elif mode in ('M5', ):
                    if 'Certain' in d_tmp['section_answer'] or 'Moderate' in d_tmp['section_answer']:
                        df_run.loc[ind, 'Toxicity Class'] = 'Most'
                    elif 'Possible' in d_tmp['section_answer']:
                        df_run.loc[ind, 'Toxicity Class'] = 'Less'
                    elif re.search('Pre-existed', d_tmp['section_answer'], re.I):
                        df_run.loc[ind, 'Toxicity Class'] = 'Precaution'
                    else:
                        continue
                else:
                    continue
                
                df_run.loc[ind, 'Evidence'] = d_tmp['section_answer']
                df_run.loc[ind, 'Supported Section'] = d_tmp['curr_section']
                df_run.loc[ind, 'SETID'] = d_tmp['reference']['SETID']
                break
                
        ## Next, check the Adverse Reactions section the last.
        if df_run.loc[ind, 'Toxicity Class'] in ('Most', 'Less'):
            continue
        if mode == 'M4':
            AR_responses = df_answer[df_answer['curr_section'].isin(['34084-4', '34088-5'])]
        else:
            AR_responses = df_answer[df_answer['curr_section']=='34084-4']    
        for _, d_tmp in AR_responses.iterrows():
            if '(Found' in d_tmp['section_answer']:
                if re.search('pre-existed', d_tmp['section_answer'], re.I):
                    df_run.loc[ind, 'Toxicity Class'] = 'Precaution'
                else:
                    df_run.loc[ind, 'Toxicity Class'] = 'Less'

                df_run.loc[ind, 'Evidence'] = d_tmp['section_answer']
                df_run.loc[ind, 'Supported Section'] = d_tmp['curr_section']
                df_run.loc[ind, 'SETID'] = d_tmp['reference']['SETID']
                break

        ## If still no evidence was found, output that as the result.
        if not df_run.loc[ind, 'Toxicity Class']:    
            df_run.loc[ind, 'SETID'] = setid
            df_run.loc[ind, 'Toxicity Class'] = 'No'
            df_run.loc[ind, 'Evidence'] = 'No Evidence was found.'
            df_run.loc[ind, 'Supported Section'] = 'No'
            
    return df_run

## Step 1. Get the last version before the update 

In [4]:
## Initial list;
latest_file = 'Label4Tox_V3/ALT_update_05052026_AI4_c.xlsx'
df_prev = pd.read_excel(latest_file).fillna('')
print(df_prev.shape)

(54840, 14)


## Step 2. Get the current complete list from FDALabel

In [ ]:
import oracledb
dsnStr = oracledb.makedsn('ncsvmlbldbtst2.fda.gov','1521','lbltst2')
con = oracledb.connect(user='', password='', dsn=dsnStr)
cursor = con.cursor()

query = f"""select l.format_group, l.set_id, l.product_names, l.PRODUCT_NORMD_GENERIC_NAMES, l.AUTHOR_ORG_NORMD_NAME ,l.eff_time
            from druglabel.dgv_sum_rx_spl l
            where l.document_type_loinc_code in ('34390-5', '34391-3', '45129-4')
                and l.format_group = 1
                and l.num_act_ingrs = 1
            order by l.format_group asc, l.eff_time desc
            """

results = cursor.execute(query)
results = results.fetchall()
df_result = pd.DataFrame(results, columns= ['FORMAT_GROUP', 'SETID', 'Product Name', 'Generic Name', 'Author Organization', 'SPL Effective Time'])
df_toupdate = df_result.drop_duplicates(['Product Name', 'Generic Name', 'Author Organization'])
df_toupdate.columns = ['PLR', 'SETID', 'Trade Name', 'Generic/Proper Name(s)', 'Author Organization', 'SPL Effective Time']
df_toupdate.shape

(14132, 6)

In [6]:
ind_update=[]
all_setids = df_prev['SETID'].values
for ind, d in df_toupdate.iterrows():
    if d['SETID'] in all_setids:
        continue
    else:
        ind_update.append(ind)
df_update = df_toupdate.loc[ind_update, :]
df_update.shape

(326, 6)

In [7]:
### Run AskFDALabel to all drugs that to update the DILI/DICT endpoint;
## This script is to run everything from beginning (for initialization)
output_folder = 'Label4Tox_V3/'
update_fh_new = 'update_06032026_c.xlsx'
drug_col = 'Trade Name' # DILIrank
ctype = {'M2': 'DILI',
         'M4': 'DICT',
         'M5': 'DIRI'}
for repeat in range(1):
    modes = ['M2', 'M4', 'M5',] # 'M2', 
    max_token = 1024
    top = 3
    for mode in modes:
        df_run = df_update.copy(deep=True)
        df_run['Toxicity Class']=''
        prompt = get_prompt(mode)
        df_run = main(df_run, drug_col, prompt, mode=mode, setid_use=True)
        df_run.to_excel(output_folder+ctype[mode]+'_'+update_fh_new, index=None)  

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [8]:
## check and extract orignal relevant content from the given labeling document of the evidences.
## save the original xml 
for ind, d in df_run.iterrows():
    setids = d['SETID']
    for setid in re.split(';', setids):
        setid = re.sub(r'\(Updated \d+\)', '', setid).strip()
        if not os.path.exists('References_V3/'+setid+'.xml'):
            query = f"""select l.set_id, l.product_names, s.spl_xml
                        from druglabel.sum_spl l
                        join druglabel.spl s on s.set_id = l.set_id
                        where l.set_id = '{setid}'
                    """
            result = cursor.execute(query).fetchone()             
            try:
                with open('References_V3/'+setid+'.xml', 'w') as f:
                    f.write(result[2])
            except Exception as e:
                continue

In [9]:
data_folder = output_folder #'AskLabelTox_Updates/'
file_suffix = update_fh_new #'update_01152025.xlsx'
df_update_dili = pd.read_excel(data_folder+'DILI_'+file_suffix).fillna('')
df_update_dict = pd.read_excel(data_folder+'DICT_'+file_suffix).fillna('')
df_update_diri = pd.read_excel(data_folder+'DIRI_'+file_suffix).fillna('')

In [10]:
def update_table(df_prev, df_update, mode = 'DILI', date = 'NONE'):
    for ind, d in df_update.iterrows():
        query = f'`Tox Type`=="{mode}" & `Trade Name`=="{d["Trade Name"]}" & `Generic/Proper Name(s)`=="{d["Generic/Proper Name(s)"]}" & `Author Organization`=="{d["Author Organization"]}"'
        tmp_df = df_prev.query(query)
        d['Changed']='No'
        if tmp_df.shape[0]:
            for tmp_ind in tmp_df.index:
                df_prev.loc[tmp_ind, 'Current'] = 'N'
                df_prev.loc[tmp_ind, 'Update_Notes'] += f" -> {d['SETID']}"
            curr_ind = tmp_df.index[0]
            if tmp_df.loc[curr_ind, 'Toxicity Class']!=d['Toxicity Class']:
                d['Changed']='Yes'
        # new labeling document needs to be added
        d['Tox Type'] = mode
        d['Update_Notes'] = f'Updated {date}'
        d['AI Summary'] = ''
        d['Current'] = 'Y'
        
        df_prev = pd.concat([df_prev, pd.DataFrame([d])], axis=0, ignore_index=True)
        
    return df_prev

In [11]:
df_prev = pd.read_excel('Label4Tox_V3/ALT_update_05052026_AI4_c.xlsx').fillna('')
df_prev['Changed'] = 'No'

In [12]:
df_prev.shape

(54840, 14)

In [13]:
date='06/03/2026'
df_curr = update_table(df_prev, df_update_dili, 'DILI', date = date)
df_curr = update_table(df_curr, df_update_dict, 'DICT', date = date)
df_curr = update_table(df_curr, df_update_diri, 'DIRI', date = date)

In [14]:
update_date='06032026'
df_curr = df_curr.reset_index(drop=True)
df_curr.to_excel(f'Label4Tox_V3/ALT_update_{update_date}_c.xlsx',index=None)

In [15]:
#df_curr = pd.read_excel(f'Label4Tox_V3/ALT_update_{update_date}_c.xlsx').filna('')
df_curr = df_curr.fillna('')
df_curr.shape

(55818, 14)

In [16]:
nms = {'xmlns':'urn:hl7-org:v3'}
abbrev = {'DICT': 'Drug-induced Cardiotoxicity', 
          'DILI': 'Drug-induced Liver Injury', 
          'DIRI': 'Drug-induced Renal Injury'
         }

for ind, d in tqdm(df_curr.iterrows()):
    drug_name = d['Trade Name']
    tox_type = d['Tox Type']
    
    # check the quality of current AI summary
    summary = d['AI Summary'].lower()
    if summary:
        continue
        
    evidence = re.sub(r'\[Severity.*','',d['Evidence'])
    xml_file = f'References_V3/{d["SETID"]}.xml'
    if not os.path.exists(xml_file):
        xml_file = f'References_V2/{d["SETID"]}.xml'
    if not os.path.exists(xml_file):
        df_curr.loc[ind, 'AI Summary'] = '[OLD VERISON] ' + df_curr.loc[ind, 'AI Summary']
        continue
    xml_data = ET.parse(xml_file).getroot()
    used_section = d['Supported Section']
    if used_section in ['34066-1','43685-7','34084-4']: # if it is from a specific section, only looked at that section
        section_data = xml_data.find(".//xmlns:section/xmlns:code[@code='"+used_section+"']/..", nms)
    else:
        section_data = xml_data
    xml_data_use = ET.tostring(section_data, method='xml',).decode('ISO8859-1')
    prompt = f'''Summarize the provided document focusing on {tox_type} with the help of the provided keywords and clues.
    {tox_type} stands for {abbrev[tox_type]}. Do not summarize other information not related to {tox_type}.
    If evidence was provided in Boxed Warnings or Warning & Precautions, quote the original context related to that {tox_type} toxicity.
    If the evidence was provided in Adverse Reactions, such as from the table, re-format the content into a short summary.
    Do Not include information irrlevant to {tox_type}.
    Format the answer into a brief report with markdown style. besides the quotes, the other part should not exceed 200 words 
'''
    message = f'''
### Keywords and severity of {tox_type} for summarization
{evidence}
### Document
{xml_data_use}
'''
    if len(message)>1000000: # in case it is too long
        message = message[:1000000] + '(...the following information is omitted due to the limitation of length allowed for the LLM)'
    try:
        answer = call_llm(message, prompt, max_token=2000)
        df_curr.loc[ind, 'AI Summary'] = answer
    except Exception as e:
        df_curr.loc[ind, 'AI Summary'] = f'Not Available; {e}'
        print(ind, e)

0it [00:00, ?it/s]

In [17]:
df_curr.to_excel(f'Label4Tox_V3/ALT_update_{update_date}_AI4_c.xlsx',index=None)
os.system(f'cp -f Label4Tox_V3/ALT_update_{update_date}_AI4_c.xlsx /main/Docker/askLabelTox/server/database/')
os.system(f'cp -f Label4Tox_V3/ALT_update_{update_date}_AI4_c.xlsx /main/Docker/askLabelTox/server/database/ALT_update_latest.xlsx') # update the latest version
os.system('cp -nr References_V3/ /main/Docker/askLabelTox/server/database/')

os.system(f'cp -f Label4Tox_V3/ALT_update_{update_date}_AI4_c.xlsx /main/Docker/askFDALabel-dev/data/downloads/ALT_update_latest.xlsx') # update the latest version
# os.system(f'cp -f Label4Tox_V3/ALT_update_{update_date}_AI4_c.xlsx /main/Docker/askFDALabel/data/downloads/ALT_update_latest.xlsx') # update the latest version
# os.system('cp -nr References_V2/. /main/Docker/askLabelTox/alt-server/database/References_V3/.')
# os.system('cp -nr References_V2/. References_V3/.')

0

In [18]:
df_curr.shape

(55818, 14)

### Re-run problematic results

In [31]:
df_dict_problematic = pd.read_excel('Demonstration/Supple Table 3 (DICT Prediction Errors).xlsx', index_col=None)
df_dict_problematic.shape

(72, 13)

In [38]:
df_curr.head(1)

,PLR,SETID,Trade Name,Generic/Proper Name(s),Author Organization,SPL Effective Time,Toxicity Class,Evidence,Supported Section,Tox Type,Update_Notes,AI Summary,Current,Changed
0,1,766b73e2-49bb-47f9-bbb1-44cfa1a3197e,BENDAMUSTINE HYDROCHLORIDE,BENDAMUSTINE HYDROCHLORIDE,BAXTER HEALTHCARE CORPORATION,20250210,Most,(Found) Hepatotoxicity; liver injury; hepatiti...,43685-7,DILI,Initialized 01/29/2025,## DILI (Drug-Induced Liver Injury) Report for...,Y,No


In [44]:
df_dict_problematic['AI Summary New']=''

In [ ]:
for ind, d in tqdm(df_dict_problematic.iterrows()):
    trade_name = d['Trade Name'].upper()
    generic_name = d['Generic/Proper Name(s)'].upper()
    df_tmp = df_curr.loc[(df_curr['Trade Name'].str.upper()==trade_name) & (df_curr['Generic/Proper Name(s)'].str.upper()==generic_name)& (df_curr['Tox Type']=='DICT')]
    df_tmp = df_tmp.sort_values(['SPL Effective Time'], ascending=False)
    
    for _, d2 in df_tmp.iterrows():
        print(ind, d['Trade Name'], d2['Trade Name'])
        setid = d2['SETID']
        evidence = d2['Evidence']
        xml_file = f'References_V3/{setid}.xml'
        if not os.path.exists(xml_file):
            xml_file = f'References_V2/{setid}.xml'
        if not os.path.exists(xml_file):
            # df_dict_problematic.loc[ind, 'AI Summary New'] = '[OLD VERISON] ' + df_dict_problematic.loc[ind, 'AI Summary New']
            print('not found', ind, setid, trade_name)
            continue
        xml_data = ET.parse(xml_file).getroot()
        section_data = xml_data
        xml_data_use = ET.tostring(section_data, method='xml',).decode('ISO8859-1')
        prompt = f'''Summarize the provided document focusing on DICT with the help of the provided keywords and clues.
        DICT stands for Drug Induced Cardiotoxicity. Do not summarize other information not related to DICT.
        If evidence was provided in Boxed Warnings or Warning & Precautions, quote the original context related to that DICT toxicity.
        If the evidence was provided in Adverse Reactions, such as from the table, re-format the content into a short summary.
        Do Not include information irrlevant to DICT.
        Format the answer into a brief report with markdown style. besides the quotes, the other part should not exceed 200 words 
    '''
        message = f'''
    ### possible keywords to support DICT:
    {evidence}
    ### Document
    {xml_data_use}
    '''
        try:
            answer = call_llm(message, prompt, max_token=2000)
            df_dict_problematic.loc[ind, 'AI Summary New'] = answer
            break
        except Exception as e:
            print(ind, e)
        

In [ ]:
df_dict_problematic['AI Summary New'].values[:2]

In [49]:
df_dict_problematic.to_excel(f'Demonstration/Supple Table 3.1.xlsx',index=None)

## Run Updating script (For Automation)

In [ ]:
import os
os.system('python NMD_update_Company.py --prev 07/21/2025 --curr 08/18/2025')